In [ ]:
!pip -q install kaggle
import zipfile, pathlib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configuraciones
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

pathlib.Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json


In [ ]:
# Descarga
!kaggle datasets download -d blastchar/telco-customer-churn -p /content

# Descomprimir
with zipfile.ZipFile('/content/telco-customer-churn.zip', 'r') as z:
    z.extractall('/content/telco-customer-churn')

!ls -lh /content/telco-customer-churn

df_churn = pd.read_csv('/content/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')


In [ ]:
# Análisis de servicios y su impacto en churn
print("ANÁLISIS DE SERVICIOS VS CHURN")

# Servicios a analizar
servicios = ['PhoneService', 'InternetService', 'OnlineSecurity', 'TechSupport', 'Contract']

# Análisis por tipo de servicio de internet
print("\n1. Análisis por Tipo de Internet:")
internet_churn = df_churn.groupby('InternetService')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).round(1)
for service, rate in internet_churn.items():
    count = len(df_churn[df_churn['InternetService'] == service])
    print(f"   {service}: {rate}% churn ({count:,} clientes)")

# Análisis por tipo de contrato
print("\n2. Análisis por Tipo de Contrato:")
contract_churn = df_churn.groupby('Contract')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).round(1)
for contract, rate in contract_churn.items():
    count = len(df_churn[df_churn['Contract'] == contract])
    print(f"   {contract}: {rate}% churn ({count:,} clientes)")

# Análisis por soporte técnico
print("\n3. Análisis por Soporte Técnico:")
tech_churn = df_churn.groupby('TechSupport')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).round(1)
for tech, rate in tech_churn.items():
    count = len(df_churn[df_churn['TechSupport'] == tech])
    print(f"   {tech}: {rate}% churn ({count:,} clientes)")

# Análisis por método de pago
print("\n4. Análisis por Método de Pago:")
payment_churn = df_churn.groupby('PaymentMethod')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).round(1)
for payment, rate in payment_churn.items():
    count = len(df_churn[df_churn['PaymentMethod'] == payment])
    print(f"   {payment}: {rate}% churn ({count:,} clientes)")


In [ ]:
# Visualizaciones de servicios
print("VISUALIZACIONES DE SERVICIOS")

# Crear subplots para diferentes análisis
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Gráfico 1: Churn por tipo de internet
internet_data = pd.crosstab(df_churn['InternetService'], df_churn['Churn'])
internet_data.plot(kind='bar', ax=axes[0,0], color=['lightblue', 'salmon'])
axes[0,0].set_title('Churn por Tipo de Servicio de Internet')
axes[0,0].set_xlabel('Tipo de Internet')
axes[0,0].set_ylabel('Número de Clientes')
axes[0,0].legend(['No Churn', 'Churn'])
axes[0,0].tick_params(axis='x', rotation=45)

# Gráfico 2: Churn por tipo de contrato
contract_data = pd.crosstab(df_churn['Contract'], df_churn['Churn'])
contract_data.plot(kind='bar', ax=axes[0,1], color=['lightgreen', 'lightcoral'])
axes[0,1].set_title('Churn por Tipo de Contrato')
axes[0,1].set_xlabel('Tipo de Contrato')
axes[0,1].set_ylabel('Número de Clientes')
axes[0,1].legend(['No Churn', 'Churn'])
axes[0,1].tick_params(axis='x', rotation=45)

# Gráfico 3: Distribución de cargos mensuales por churn
axes[1,0].hist([df_churn[df_churn['Churn']=='No']['MonthlyCharges'], 
                df_churn[df_churn['Churn']=='Yes']['MonthlyCharges']], 
               bins=20, alpha=0.7, label=['No Churn', 'Churn'])
axes[1,0].set_title('Distribución de Cargos Mensuales')
axes[1,0].set_xlabel('Cargos Mensuales')
axes[1,0].set_ylabel('Frecuencia')
axes[1,0].legend()

# Gráfico 4: Churn por método de pago
payment_rates = df_churn.groupby('PaymentMethod')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
payment_rates.plot(kind='bar', ax=axes[1,1], color='orange')
axes[1,1].set_title('Tasa de Churn por Método de Pago')
axes[1,1].set_xlabel('Método de Pago')
axes[1,1].set_ylabel('Tasa de Churn (%)')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# Análisis de combinaciones de servicios
print("ANÁLISIS DE COMBINACIONES DE SERVICIOS")

# Crear índice de servicios adicionales
def contar_servicios_adicionales(row):
    servicios_extra = 0
    
    if row['OnlineSecurity'] == 'Yes':
        servicios_extra += 1
    if row['TechSupport'] == 'Yes':
        servicios_extra += 1
    if 'MultipleLines' in df_churn.columns and row['MultipleLines'] == 'Yes':
        servicios_extra += 1
    
    return servicios_extra

df_churn['servicios_adicionales'] = df_churn.apply(contar_servicios_adicionales, axis=1)

# Análisis por cantidad de servicios
servicios_churn = df_churn.groupby('servicios_adicionales')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).round(1)

print("\nTasa de churn por cantidad de servicios adicionales:")
for num_servicios, rate in servicios_churn.items():
    count = len(df_churn[df_churn['servicios_adicionales'] == num_servicios])
    print(f"  {num_servicios} servicios: {rate}% churn ({count:,} clientes)")

# Análisis de segmentación por valor
print(f"\nSegmentación por valor mensual:")
df_churn['segmento_valor'] = pd.cut(df_churn['MonthlyCharges'], 
                                   bins=3, 
                                   labels=['Bajo', 'Medio', 'Alto'])

valor_churn = df_churn.groupby('segmento_valor')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).round(1)
for segmento, rate in valor_churn.items():
    count = len(df_churn[df_churn['segmento_valor'] == segmento])
    print(f"  Valor {segmento}: {rate}% churn ({count:,} clientes)")

# Visualización final
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico 1: Churn por cantidad de servicios
servicios_rates = df_churn.groupby('servicios_adicionales')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
servicios_rates.plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Tasa de Churn por Cantidad de Servicios Adicionales')
axes[0].set_xlabel('Número de Servicios Adicionales')
axes[0].set_ylabel('Tasa de Churn (%)')
axes[0].tick_params(axis='x', rotation=0)

# Gráfico 2: Churn por segmento de valor
valor_rates = df_churn.groupby('segmento_valor')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
valor_rates.plot(kind='bar', ax=axes[1], color=['green', 'orange', 'red'])
axes[1].set_title('Tasa de Churn por Segmento de Valor')
axes[1].set_xlabel('Segmento de Valor')
axes[1].set_ylabel('Tasa de Churn (%)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

# Conclusiones
print(f"\nCONCLUSIONES PRINCIPALES:")
min_churn_servicios = servicios_churn.idxmin()
max_churn_servicios = servicios_churn.idxmax()
print(f"- Clientes con {min_churn_servicios} servicios tienen menor churn ({servicios_churn[min_churn_servicios]:.1f}%)")
print(f"- Clientes con {max_churn_servicios} servicios tienen mayor churn ({servicios_churn[max_churn_servicios]:.1f}%)")

contract_min = contract_churn.idxmin()
contract_max = contract_churn.idxmax()
print(f"- Contratos '{contract_min}' tienen menor churn ({contract_churn[contract_min]:.1f}%)")
print(f"- Contratos '{contract_max}' tienen mayor churn ({contract_churn[contract_max]:.1f}%)")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configuración
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (8, 6)

print("Iniciando análisis de comportamiento de clientes...")

# Cargar datos
try:
    data = pd.read_csv('music_data_cleaned.csv')
    print(f"Datos cargados: {data.shape[0]} registros, {data.shape[1]} columnas")
    
    # Información básica
    print("\nInformación del dataset:")
    print(data.info())
    
except Exception as e:
    print(f"Error: {e}")
    data = pd.DataFrame()


In [ ]:
# Análisis descriptivo
if not data.empty:
    print("ANÁLISIS DESCRIPTIVO")
    
    # Seleccionar columnas categóricas y numéricas
    categorical_cols = data.select_dtypes(include=['object']).columns.tolist()
    numerical_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    
    print(f"\nColumnas categóricas: {len(categorical_cols)}")
    print(categorical_cols[:3])
    print(f"\nColumnas numéricas: {len(numerical_cols)}")
    print(numerical_cols[:3])
    
    # Estadísticas básicas para columnas numéricas
    if numerical_cols:
        print("\nEstadísticas descriptivas:")
        stats = data[numerical_cols[:2]].describe()
        print(stats)
        
        # Correlación entre variables numéricas
        if len(numerical_cols) >= 2:
            corr = data[numerical_cols[:3]].corr()
            print(f"\nMatriz de correlación:")
            print(corr.round(2))
    
    # Análisis de valores únicos para categóricas
    if categorical_cols:
        print(f"\nAnálisis de variables categóricas:")
        for col in categorical_cols[:2]:
            unique_count = data[col].nunique()
            print(f"  {col}: {unique_count} valores únicos")
            if unique_count <= 10:
                print(f"    Valores: {data[col].value_counts().index.tolist()}")
                
else:
    print("No hay datos para analizar")


In [ ]:
# Visualización de patrones
if not data.empty:
    print("VISUALIZACIÓN DE PATRONES")
    
    # Crear visualizaciones básicas
    if len(numerical_cols) >= 2:
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        
        # Histograma de primera variable numérica
        data[numerical_cols[0]].hist(bins=20, ax=axes[0,0])
        axes[0,0].set_title(f'Distribución de {numerical_cols[0]}')
        axes[0,0].set_xlabel(numerical_cols[0])
        axes[0,0].set_ylabel('Frecuencia')
        
        # Histograma de segunda variable numérica
        data[numerical_cols[1]].hist(bins=20, ax=axes[0,1])
        axes[0,1].set_title(f'Distribución de {numerical_cols[1]}')
        axes[0,1].set_xlabel(numerical_cols[1])
        axes[0,1].set_ylabel('Frecuencia')
        
        # Scatter plot
        axes[1,0].scatter(data[numerical_cols[0]], data[numerical_cols[1]], alpha=0.5)
        axes[1,0].set_title(f'{numerical_cols[0]} vs {numerical_cols[1]}')
        axes[1,0].set_xlabel(numerical_cols[0])
        axes[1,0].set_ylabel(numerical_cols[1])
        
        # Boxplot de primera variable
        data[numerical_cols[0]].plot(kind='box', ax=axes[1,1])
        axes[1,1].set_title(f'Boxplot de {numerical_cols[0]}')
        axes[1,1].set_ylabel(numerical_cols[0])
        
        plt.tight_layout()
        plt.show()
    
    # Análisis de variables categóricas
    if categorical_cols:
        # Tomar primera variable categórica para análisis
        cat_var = categorical_cols[0]
        value_counts = data[cat_var].value_counts()
        
        print(f"\nDistribución de {cat_var}:")
        for val, count in value_counts.head(5).items():
            pct = count / len(data) * 100
            print(f"  {val}: {count} ({pct:.1f}%)")
        
        # Gráfico de barras
        plt.figure(figsize=(10, 6))
        value_counts.head(8).plot(kind='bar')
        plt.title(f'Distribución de {cat_var}')
        plt.xlabel(cat_var)
        plt.ylabel('Frecuencia')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        
else:
    print("No hay datos para visualizar")


In [ ]:
# Análisis comparativo simple
if not data.empty and len(numerical_cols) >= 1 and len(categorical_cols) >= 1:
    print("ANÁLISIS COMPARATIVO")
    
    # Tomar una variable categórica para agrupar
    group_var = categorical_cols[0]
    numeric_var = numerical_cols[0]
    
    # Obtener los grupos más comunes (máximo 5)
    top_groups = data[group_var].value_counts().head(5).index.tolist()
    data_filtered = data[data[group_var].isin(top_groups)]
    
    print(f"\nAnálisis de {numeric_var} por {group_var}:")
    
    # Estadísticas por grupo
    group_stats = data_filtered.groupby(group_var)[numeric_var].agg(['count', 'mean', 'median']).round(2)
    print("\nEstadísticas por grupo:")
    print(group_stats)
    
    # Visualización comparativa
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Boxplot por grupo
    data_filtered.boxplot(column=numeric_var, by=group_var, ax=axes[0])
    axes[0].set_title(f'{numeric_var} por {group_var}')
    axes[0].set_xlabel(group_var)
    axes[0].set_ylabel(numeric_var)
    
    # Barplot de medias
    group_means = data_filtered.groupby(group_var)[numeric_var].mean()
    group_means.plot(kind='bar', ax=axes[1])
    axes[1].set_title(f'Media de {numeric_var} por {group_var}')
    axes[1].set_xlabel(group_var)
    axes[1].set_ylabel(f'Media de {numeric_var}')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Insights básicos
    best_group = group_means.idxmax()
    worst_group = group_means.idxmin()
    
    print(f"\nInsights:")
    print(f"Grupo con mayor {numeric_var}: {best_group} ({group_means[best_group]:.2f})")
    print(f"Grupo con menor {numeric_var}: {worst_group} ({group_means[worst_group]:.2f})")
    
else:
    print("No hay suficientes variables para análisis comparativo")
